# 01_prep — Dev 데이터 만들기 (담당: core)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Walnutbrown/kdt14th-lendingclub/blob/main/notebooks/01_prep.ipynb)

입력: `raw/lending_club_2020_train_new.csv`, `raw/us_treasury_3y_5y_monthly.csv`
출력: `data/dev_X_raw.parquet`, `data/dev_meta.parquet`, `contract/` (범주·열 순서·부호), `report/` (요약표)

- 여기서 하는 것은 **행마다 고정 규칙(Stage 1)**과 **y를 쓰지 않는 약속 정하기**뿐입니다.
- 대치·윈저라이징·스케일링(Stage 2)은 하지 않습니다. 모형 노트북이 부트스트랩 회차마다 Train_b에서 합니다.
- 4번 칸의 전처리기 fit은 열 이름과 순서를 얻기 위한 것이고, 모형 학습에 쓰지 않습니다.

In [5]:
import sys
if "google.colab" in sys.modules:
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
import time, datetime

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[colab_setup] 브랜치 main (fda9f461) | raw·data·outputs·report·freeze·test_hold → /content/drive/MyDrive/SNU KDT 14 4조/LendingClub | 위치 /content/kdt14th-lendingclub/notebooks
[config 2026-11-26] 공유해시 d672cc3ba054 | 전체해시 0ec794b17ddd (logit 24f0c250f0e9 | xgb 56c27c8d0e0e | test a404a78b5fa3) | STAGE=dev | 미결 3개 (B_TEST, DELTA, XGB_PARAMS), 임시값으로 실행 중
[lib_prep] 해시 e07ddd6943c5


In [6]:
# 1. Stage 1: 표본 필터 → 국채 매칭·MIRR → 37개 변수 파싱
t0 = time.time()
raw_X, meta = load_and_prepare(TRAIN_CSV, TREASURY_CSV, is_labeled=True, **SAMPLE)
print(f"{time.time() - t0:.0f}초 | raw_X {raw_X.shape} | meta {meta.shape}")

99초 | raw_X (1109665, 37) | meta (1109665, 12)


In [7]:
# 2. 검증 (구조·알고리즘 불변식은 항상 assert, EXPECT 수치 일치는 STAGE='final'에서만 강제)
n, dr = len(raw_X), meta["y"].mean()
checks = {
    "행 수 > 0": n > 0,
    "부도율 범위 (0, 1)": 0 < dr < 1,
    f"변수 {EXPECT['n_raw_features']}개": raw_X.shape[1] == EXPECT["n_raw_features"],
    "X와 meta 행 수 같음": len(meta) == n,
    "y는 0/1만 (미종결 없음)": set(meta["y"].unique()) <= {0, 1},
    "id 중복 없음": meta["id"].is_unique,
    "국채 매칭 실패 없음": meta["rf_ann"].notna().all(),
    "MIRR 결측·무한대 없음": np.isfinite(meta["mirr"]).all(),
    "금지 칼럼이 X에 없음": not (FORBIDDEN_COLS & set(raw_X.columns)),
    "행 번호 0..N-1": raw_X.index.equals(pd.RangeIndex(n)) and meta.index.equals(pd.RangeIndex(n)),
}
expect_warns = []
if EXPECT["n_dev"] is not None:
    ok_n = (n == EXPECT["n_dev"])
    if STAGE == "final":
        checks[f"행 수 = {EXPECT['n_dev']:,}"] = ok_n
    elif not ok_n:
        expect_warns.append(f"행 수 {n:,} ≠ EXPECT {EXPECT['n_dev']:,}")
if EXPECT["default_rate"] is not None:
    ok_dr = (round(dr, 4) == EXPECT["default_rate"])
    if STAGE == "final":
        checks[f"부도율 = {EXPECT['default_rate']:.2%}"] = ok_dr
    elif not ok_dr:
        expect_warns.append(f"부도율 {round(dr, 4):.4f} ≠ EXPECT {EXPECT['default_rate']:.4f}")
for k, ok in checks.items():
    print("PASS" if ok else "FAIL", k)
print(f"실제: {n:,}행, 부도율 {dr:.4%}")
if expect_warns:
    print("경고(STAGE != 'final'이라 계속 진행 — 호환/소형 입력 개발 검증 허용):", "; ".join(expect_warns))
if EXPECT["n_dev"] is None or EXPECT["default_rate"] is None:
    print(f"[00_config EXPECT 업데이트용] n_dev={n:_}, default_rate={round(dr, 4)}")
assert all(checks.values()), "검증 실패. 위 FAIL 항목 확인"

PASS 행 수 > 0
PASS 부도율 범위 (0, 1)
PASS 변수 37개
PASS X와 meta 행 수 같음
PASS y는 0/1만 (미종결 없음)
PASS id 중복 없음
PASS 국채 매칭 실패 없음
PASS MIRR 결측·무한대 없음
PASS 금지 칼럼이 X에 없음
PASS 행 번호 0..N-1
실제: 1,109,665행, 부도율 16.2636%


In [8]:
# 3. 저장 후 다시 읽어 같은지 확인 + 전처리 이력(prep_meta.json) 기록
raw_X.to_parquet(DATA_DIR / "dev_X_raw.parquet", index=False)
meta.to_parquet(DATA_DIR / "dev_meta.parquet", index=False)
assert pd.read_parquet(DATA_DIR / "dev_X_raw.parquet").equals(raw_X)
assert pd.read_parquet(DATA_DIR / "dev_meta.parquet").equals(meta)

PREP_META = dict(
    created=datetime.datetime.now().isoformat(timespec="seconds"),
    stage=STAGE,
    n_rows=int(len(raw_X)),
    default_rate=round(float(meta["y"].mean()), 6),
    train_csv=TRAIN_CSV.name,
    train_csv_sha256=sha256_file(TRAIN_CSV),
    treasury_csv=TREASURY_CSV.name,
    treasury_csv_sha256=sha256_file(TREASURY_CSV),
    data_files={name: sha256_file(DATA_DIR / name) for name in ("dev_X_raw.parquet", "dev_meta.parquet")},
    sample=SAMPLE,
    prep=PREP,
    prep_hash=config_fingerprint("prep"),
    config_hash=config_fingerprint("shared"),
    lib_prep_hash=LIB_PREP_HASH,
    meta=run_meta(),
)
(DATA_DIR / "prep_meta.json").write_text(json.dumps(PREP_META, ensure_ascii=False, indent=1), encoding="utf-8")
print("저장:", [p.name for p in sorted(DATA_DIR.glob("dev_*.parquet"))], "| prep_meta.json")
print("dev sha256:", {k: v[:12] for k, v in PREP_META["data_files"].items()})

저장: ['dev_X_raw.parquet', 'dev_meta.parquet'] | prep_meta.json
dev sha256: {'dev_X_raw.parquet': '952f319a18c1', 'dev_meta.parquet': '3bd3a51532d0'}


In [9]:
# 4. 약속 파일 (y 사용 안 함): 범주 목록 → 열 순서 → 부호
cats = rare_category_levels(raw_X)
(CONTRACT_DIR / "categories.json").write_text(json.dumps(cats, ensure_ascii=False, indent=2), encoding="utf-8")

cols = {}
for kind, fname in (("xgb", "columns_xgb.json"), ("logit", "columns_logit.json")):
    ct = make_preprocessor(kind, categories=cats).fit(raw_X)
    Xt = ct.transform(raw_X)
    assert Xt.isna().sum().sum() == 0, f"{kind}: 전처리 후 결측이 남음"
    cols[kind] = list(Xt.columns)
    (CONTRACT_DIR / fname).write_text(json.dumps(cols[kind], ensure_ascii=False, indent=2), encoding="utf-8")
assert cols["xgb"] == cols["logit"], "xgb와 logit의 열이 다름"

signs = dict(zip(cols["xgb"], mono_constraints(ct)))          # 열 이름 기준, 금지 칼럼 검사 포함
(CONTRACT_DIR / "signs.json").write_text(json.dumps(signs, ensure_ascii=False, indent=2), encoding="utf-8")

PREP_META["contract_hashes"] = contract_hashes()
(DATA_DIR / "prep_meta.json").write_text(json.dumps(PREP_META, ensure_ascii=False, indent=1), encoding="utf-8")

n_na  = sum(c.endswith("_na") for c in cols["xgb"])
n_ohe = sum(c.startswith(tuple(cats)) for c in cols["xgb"])
print(f"모형 입력 {len(cols['xgb'])}열 = 수치 {len(cols['xgb']) - n_na - n_ohe} + 결측지시 {n_na} + 원핫 {n_ohe}")
print("범주:", {k: len(v) for k, v in cats.items()}, "| 부호:", {s: list(signs.values()).count(s) for s in (-1, 0, 1)})

모형 입력 49열 = 수치 35 + 결측지시 2 + 원핫 12
범주: {'home_ownership': 4, 'purpose': 10} | 부호: {-1: 9, 0: 29, 1: 11}


In [10]:
# 5. VIF (로지스틱 입력 기준, 실험 문서 Step 2.2: < 10, 01_eda_audit 기준: < 5)
vif = compute_vif(Xt).sort_values(ascending=False)          # Xt: 4번 칸 마지막(logit) 결과
vif.round(2).to_frame().to_csv(REPORT_DIR / "table_vif.csv")
print(f"최대 VIF {vif.max():.2f} ({vif.idxmax()})")
assert vif.max() < 10, "VIF 10 이상 변수가 있음"
vif.head(5).round(2)

최대 VIF 4.45 (total_acc)


,VIF
total_acc,4.45
log_total_bal_ex_mort,4.35
log_tot_cur_bal,3.71
revol_util,3.61
open_acc,3.56


In [11]:
# 6. 보고서용 요약표 (y는 기술통계로만 사용)
yr = meta.groupby("issue_year").agg(건수=("y", "size"), 부도율=("y", "mean"), 평균초과수익률=("excess_ret", "mean"))
st = meta.groupby("y")[["rf_ann", "mirr", "excess_ret"]].agg(["mean", "median", "std"])
miss = raw_X.isna().mean().sort_values(ascending=False).rename("결측률")
yr.round(4).to_csv(REPORT_DIR / "table_year.csv")
st.round(4).to_csv(REPORT_DIR / "table_return_by_status.csv")
miss.round(4).to_csv(REPORT_DIR / "table_missing.csv")
print("저장: table_year.csv, table_return_by_status.csv, table_missing.csv, table_vif.csv")
yr.round(4)

저장: table_year.csv, table_return_by_status.csv, table_missing.csv, table_vif.csv


,건수,부도율,평균초과수익률
issue_year,,,
2013,134814,0.1560,0.0243
2014,235619,0.1772,0.0156
2015,322419,0.1640,0.0082
2016,323436,0.1588,0.0062
2017,93377,0.1441,0.0077
